# Sesión 0.7: Sets y Operaciones de Conjuntos en Alta Velocidad
## Diplomado en Ciencia de Datos e Inteligencia Artificial - INT210

---

### 1. La Analogía Infantil: El Álbum de Figuritas y los Círculos de Amigos

Imagina que estás llenando un álbum de figuritas de superhéroes:

- En tu álbum solo puedes pegar **una sola vez** cada estampa numerada. Si compras un sobre y viene una figurita repetida, simplemente no tiene espacio nuevo en el álbum. Así funciona un `set` en Python: **elimina automáticamente todos los duplicados**.
- Si tú tienes un álbum $A$ y tu mejor amigo tiene el álbum $B$:
  * Las estampas que **ambos tienen repetidas** para intercambiar forman la **Intersección ($A \cap B$)**.
  * La colección de **todas las estampas distintas** entre los dos forma la **Unión ($A \cup B$)**.
  * Las que **tú tienes pero a tu amigo le faltan** forman la **Diferencia ($A \setminus B$)**.

Comprobar si una estampa está en el álbum toma un solo segundo porque sabes exactamente la página numerada donde buscar.

---
### 2. Rigor Científico: Teoría de Conjuntos y Complejidad $\mathcal{O}(1)$ vs $\mathcal{O}(N)$ (Joel Grus, Cap. 2)

Un `set` en Python se implementa mediante una tabla hash que solo almacena claves (sin valores asociados). Esto confiere dos propiedades formales cruciales:

1. **Unicidad y Desorden:** No existen elementos duplicados y no se garantiza el orden posicional (no permite indexación directa `s[0]`).
2. **Pertenencia Instantánea:** La operación `elemento in conjunto` requiere tiempo constante promedio **$\mathcal{O}(1)$**, a diferencia de una lista donde la búsqueda secuencial toma **$\mathcal{O}(N)$**.

#### Operaciones Formales de Conjuntos:
- **Unión:** $A \cup B$ (sintaxis: `A | B` o `A.union(B)`)
- **Intersección:** $A \cap B$ (sintaxis: `A & B` o `A.intersection(B)`)
- **Diferencia:** $A \setminus B$ (sintaxis: `A - B` o `A.difference(B)`)
- **Diferencia Simétrica:** $A \Delta B = (A \setminus B) \cup (B \setminus A)$ (sintaxis: `A ^ B` o `A.symmetric_difference(B)`)

In [ ]:
# Demostración de Operaciones de Conjuntos con Conjuntos de Seguridad
ips_firewall_norte = {'192.168.1.1', '10.0.0.5', '198.51.100.4', '203.0.113.10'}
ips_firewall_sur   = {'198.51.100.4', '203.0.113.10', '172.16.0.8', '192.168.1.99'}

union_ips = ips_firewall_norte | ips_firewall_sur
interseccion_ips = ips_firewall_norte & ips_firewall_sur
solo_norte = ips_firewall_norte - ips_firewall_sur
diferencia_simetrica = ips_firewall_norte ^ ips_firewall_sur

print(f"Unión de IPs Monitoreadas (Total Únicas: {len(union_ips)}): {union_ips}")
print(f"Intersección (IPs coincidentes en ambos firewalls): {interseccion_ips}")
print(f"Diferencia (IPs exclusivas del Firewall Norte): {solo_norte}")
print(f"Diferencia Simétrica (IPs que no se repiten entre zonas): {diferencia_simetrica}")

#### Explicación Técnica Línea por Línea del Bloque de Código:

- `ips_firewall_norte = {...}`: Instancia un conjunto en memoria calculando los valores hash de cada cadena IP e insertándolas en la tabla.
- `ips_firewall_norte | ips_firewall_sur`: Ejecuta la unión matemática de conjuntos, generando un nuevo `set` con todos los elementos únicos combinados.
- `ips_firewall_norte & ips_firewall_sur`: Ejecuta la intersección hash, reteniendo únicamente las IPs presentes simultáneamente en ambos conjuntos en tiempo $\mathcal{O}(\min(|A|, |B|))$.
- `ips_firewall_norte - ips_firewall_sur`: Calcula la diferencia relativa eliminando del primer conjunto cualquier elemento que exista en el segundo.
- `ips_firewall_norte ^ ips_firewall_sur`: Calcula la diferencia simétrica (elementos que pertenecen a uno u otro conjunto, pero no a ambos a la vez).

---
### 3. Caso de Estudio 1: Filtrado Perimetral de Botnets en Ciberseguridad (Benchmark $\mathcal{O}(1)$ vs $\mathcal{O}(N)$)

Comparamos la velocidad de verificación de 100,000 conexiones entrantes contra una Lista Negra de 10,000 IPs maliciosas almacenadas en una lista frente a un conjunto.

In [ ]:
import time

# Simulación estructurada de listas negras e IPs entrantes
lista_negra_raw = [f"198.51.100.{i}" for i in range(5000)]
set_negro = set(lista_negra_raw)

ip_prueba_existente = "198.51.100.4999"   # Último elemento (peor caso para lista)
ip_prueba_inexistente = "10.0.0.1"          # No existe en la lista

# Benchmark en Lista O(N)
inicio_l = time.perf_counter()
for _ in range(1000):
    _ = ip_prueba_existente in lista_negra_raw
tiempo_lista = time.perf_counter() - inicio_l

# Benchmark en Set O(1)
inicio_s = time.perf_counter()
for _ in range(1000):
    _ = ip_prueba_existente in set_negro
tiempo_set = time.perf_counter() - inicio_s

print(f"Tiempo 1,000 búsquedas en Lista O(N): {tiempo_lista*1000:.4f} ms")
print(f"Tiempo 1,000 búsquedas en Set   O(1): {tiempo_set*1000:.4f} ms")
print(f"Factor de Aceleración: {tiempo_lista / tiempo_set:.1f}x veces más rápido")

#### Explicación Técnica Línea por Línea:
- `set(lista_negra_raw)`: Convierte la lista en un conjunto hash en una sola pasada $\mathcal{O}(N)$.
- `ip_prueba_existente in lista_negra_raw`: En la lista, Python debe iterar secuencialmente los 5,000 elementos comprobando la igualdad uno a uno en el peor caso.
- `ip_prueba_existente in set_negro`: En el conjunto, calcula `hash(ip_prueba)` y salta directamente a la cubeta de memoria correspondiente en un solo ciclo de reloj.

---
### 4. Caso de Estudio 2: Extracción y Deduplicación de Paletas Cromáticas en Arte Digital

Extrae el conjunto exacto de colores únicos presentes en una textura gráfica de renderizado representada como una lista de tuplas RGB.

In [ ]:
# Lista de píxeles con alta redundancia cromática de un lienzo digital
lienzo_pixeles = [
    (255, 0, 0), (255, 0, 0), (0, 255, 0), (0, 0, 255),
    (255, 0, 0), (0, 255, 0), (255, 255, 255), (0, 0, 0),
    (255, 0, 0), (0, 0, 255), (255, 255, 255), (255, 0, 0)
]

paleta_unica = set(lienzo_pixeles)

print(f"Total de píxeles procesados: {len(lienzo_pixeles)}")
print(f"Total de colores únicos en la paleta: {len(paleta_unica)}")
print(f"Paleta Cromática Única: {paleta_unica}")

#### Explicación Técnica Línea por Línea:
- `lienzo_pixeles`: Lista que contiene tuplas `(r, g, b)`. Dado que las tuplas son inmutables, son objetos *hashables* válidos para residir en un `set`.
- `paleta_unica = set(lienzo_pixeles)`: Deduplica automáticamente los 12 píxeles reduciéndolos a los 5 colores primarios únicos del lienzo.

---
### 5. Ejercicio Práctico Guiado: Detección de Compromiso Perimetral Cruzado (`# TODO`)

**Instrucción:** Completa la función `auditar_brecha_seguridad` para que retorne el conjunto de IPs que están en la lista negra global Y que además tuvieron al menos una conexión exitosa registrada.

In [ ]:
def auditar_brecha_seguridad(ips_exitosas: list, blacklist_global: set) -> set:
    """
    Retorna el conjunto de IPs comprometidas mediante operaciones nativas de conjuntos.
    """
    ### TU CÓDIGO AQUÍ
    pass

# Comprobación del ejercicio
# exitosas = ['10.0.0.1', '198.51.100.4', '192.168.1.5', '203.0.113.88']
# blacklist = {'198.51.100.4', '203.0.113.88', '198.51.100.99'}
# brecha = auditar_brecha_seguridad(exitosas, blacklist)
# print(f"IPs con brecha de seguridad confirmada: {brecha}")  # Debe ser {'198.51.100.4', '203.0.113.88'}